# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
dbutils.widgets.text("ruta_csv", "file:/Workspace/ventas_conflicto.csv")
ruta = dbutils.widgets.get("ruta_csv")

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

In [0]:
#Esta intentando leer el archivo pero este no existe, se podría cambiar por este codigo

df = spark.read.csv(
    "file:/Workspace/ventas_export.csv",
    header=True,
    inferSchema=True
)

display(df)


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

In [0]:
#El error está en que el pipeline intenta crear la vista oro_roto copiando todos los registros de bronce_ventas mediante SELECT *.

Esto genera un flujo incorrecto porque no se aplica ningún filtro ni transformación para separar únicamente la información que debe pasar a la capa Oro

spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_ventas AS
SELECT DISTINCT
    id_venta,
    fecha,
    id_tienda,
    total
FROM bronce_ventas
WHERE total IS NOT NULL
""")

display(spark.sql("""
SELECT COUNT(*) AS filas_oro
FROM oro_ventas
"""))

## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# Escriban aquí el flujo. No usen el CSV editado.
# spark.sql("create or replace temp view oro_fact_ventas as ...")
# spark.sql("create or replace temp view cuarentena_ventas as ...")
raise NotImplementedError("Aún no hay pipeline")

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS
SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago
FROM bronce_ventas
WHERE venta_id IS NOT NULL
  AND fecha IS NOT NULL
  AND tienda_id IS NOT NULL
  AND producto_id IS NOT NULL
  AND cantidad > 0
  AND monto > 0
  AND medio_pago IS NOT NULL
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS
SELECT
    *,
    CASE
        WHEN venta_id IS NULL THEN 'venta_id nulo'
        WHEN fecha IS NULL THEN 'fecha nula'
        WHEN tienda_id IS NULL THEN 'tienda_id nulo'
        WHEN producto_id IS NULL THEN 'producto_id nulo'
        WHEN cantidad IS NULL OR cantidad <= 0 THEN 'cantidad inválida'
        WHEN monto IS NULL OR monto <= 0 THEN 'monto inválido'
        WHEN medio_pago IS NULL THEN 'medio_pago nulo'
        ELSE 'Registro válido'
    END AS motivo
FROM bronce_ventas
WHERE venta_id IS NULL
   OR fecha IS NULL
   OR tienda_id IS NULL
   OR producto_id IS NULL
   OR cantidad IS NULL
   OR cantidad <= 0
   OR monto IS NULL
   OR monto <= 0
   OR medio_pago IS NULL
""")

In [0]:
display(spark.sql("SELECT * FROM oro_fact_ventas"))

In [0]:
display(spark.sql("SELECT * FROM cuarentena_ventas"))

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS
SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    CAST(monto AS DOUBLE) AS monto,
    medio_pago
FROM bronce_ventas
WHERE venta_id IS NOT NULL
  AND fecha IS NOT NULL
  AND try_cast(tienda_id AS INT) IN (1, 2, 3)
  AND producto_id IS NOT NULL
  AND cantidad IS NOT NULL
  AND cantidad > 0
  AND try_cast(monto AS DOUBLE) IS NOT NULL
  AND medio_pago IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY venta_id
    ORDER BY fecha
) = 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS
SELECT
    *,
    CASE
        WHEN venta_id IS NULL THEN 'venta_id nulo'
        WHEN fecha IS NULL THEN 'fecha nula'
        WHEN try_cast(tienda_id AS INT) NOT IN (1, 2, 3)
             OR tienda_id IS NULL THEN 'tienda inválida'
        WHEN producto_id IS NULL THEN 'producto_id nulo'
        WHEN cantidad IS NULL OR cantidad <= 0 THEN 'cantidad inválida'
        WHEN try_cast(monto AS DOUBLE) IS NULL THEN 'monto no numérico'
        WHEN medio_pago IS NULL THEN 'medio_pago nulo'
        ELSE 'Registro válido'
    END AS motivo
FROM bronce_ventas
WHERE venta_id IS NULL
   OR fecha IS NULL
   OR try_cast(tienda_id AS INT) NOT IN (1, 2, 3)
   OR tienda_id IS NULL
   OR producto_id IS NULL
   OR cantidad IS NULL
   OR cantidad <= 0
   OR try_cast(monto AS DOUBLE) IS NULL
   OR medio_pago IS NULL
""")

## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
2. ¿Qué hicieron con un monto que venía en texto?
3. ¿Por qué una tienda se quedó fuera?
4. ¿Dónde quedó la columna que no estaba en el contrato?
5. Si corren el notebook otra vez, ¿el hecho crece?

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio

In [0]:
1. **¿Qué pasó con la misma venta llegada dos veces?**
   Si la misma venta llegó dos veces, dejamos solamente una en la tabla de Oro. Así evitamos que la misma venta quede repetida.

2. **¿Qué hicieron con un monto que venía en texto?**
   Revisamos si el monto se podía convertir a número. Si no se podía convertir, esa venta no entraba a Oro y se mandaba a cuarentena.

3. **¿Por qué una tienda se quedó fuera?**
   Porque solo se aceptaron las tiendas 1, 2 y 3. Si una venta tenía otra tienda, no cumplía con las condiciones y se iba a cuarentena.

4. **¿Dónde quedó la columna que no estaba en el contrato?**
   La columna `canal` no se pasó a Oro, porque no estaba dentro de las columnas que se habían definido para la tabla de ventas.

5. **Si corren el notebook otra vez, ¿el hecho crece?**
   No. Como estamos creando nuevamente la vista con las mismas reglas, no se van acumulando los datos ni se duplican las ventas cada vez que se ejecuta el notebook.


In [0]:
print("listo")